In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [72]:
df = pd.read_csv('train.txt',sep=';',header= None, names=['text','emotions'])

In [73]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [74]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [75]:
df['emotions'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [76]:
unique_emotions = df['emotions'].unique()
emotion_numbers = {}
i = 0
for emotions in unique_emotions:
    emotion_numbers[emotions] = i
    i += 1

df['emotions'] = df['emotions'].map(emotion_numbers)

Using LabelEncoder

In [77]:
# from sklearn.preprocessing import LabelEncoder
# encoder = LabelEncoder()
# df['emotions'] = encoder.fit_transform(df['emotions'])

To see the mapping

In [78]:
# mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
# print(mapping)

To convert predictions back to emotion names

In [79]:
# predicted_emotions = encoder.inverse_transform([0, 1, 2, 3, 4, 5])
# print(predicted_emotions)

In [80]:
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


LowerCase

In [81]:
df['text'] = df['text'].apply(lambda x : x.lower())
df.head()

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


Remove Punctuations

In [82]:
import string

def remove_punc(txt):
    return txt.translate((str.maketrans('','',string.punctuation)))

In [83]:
df['text'] = df['text'].apply(remove_punc)

Remove Numbers

In [84]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i   # It will ignore the digits and will only add string values
    return new

df['text'] = df['text'].apply(remove_numbers)

Remove URLs/Links and HTML Tags

In [85]:
# For students

Remove Emojis

In [86]:
# For this we will only keep the ASCII values

def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

Remove Stopwords (NLTK/SpaCy)

In [87]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [88]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhart\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhart\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [89]:
# It have stopwords for many languages like French, russian, bengali, etc.

stop_words = set(stopwords.words('english'))
stop_words
len(stop_words)

198

In [90]:
df.loc[1]
df.loc[1]['text']

# We will perform tokenization 

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

## **Difference btw Tokenization & Splitting**

Splitting means breaking text into pieces using a simple separator.
``` python
text = "I love NLP"
words = text.split()
print(words)    # Ouput=['I','love','NLP']
```

Tokenization means breaking text into meaningful NLP units called tokens, usually words or punctuation.

``` python
from nltk.tokenize import word_tokenize
text = "I love NLP!"
tokens = word_tokenize(text)
print(tokens)   #Ouput=['I', 'love', 'NLP', '!']
text.split()    #Output=['I', 'love', 'NLP!']
```

In [91]:
def remove(txt):
    words = word_tokenize(txt)
    # words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove)

In [92]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [93]:
df.head()

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [94]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotions'], test_size=0.2, random_state=42)

In [95]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

### **Bag of Words**

In [96]:
bow_vectorizer = CountVectorizer()

In [ ]:
X_train_bow = bow_vectorizer.fit_transform(X_train)     # Fit_transform: learn the vocabulary and word statistics from the training text and convert the training text into numerical vectors.
# X_test_bow = bow_vectorizer.fit_transform(X_test)     # Causes error on model.predict(X_test_bow)
X_test_bow = bow_vectorizer.transform(X_test)   # Transform: The test data must be converted using the same vocabulary learned from the training data

# In short:

# Training data: learn + convert
# Test data: convert using what was learned

In [98]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [99]:
X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 26934 stored elements and shape (3200, 13359)>

#### **Multinomial Naive Bayes**

In [100]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [101]:
nb_model_bow = MultinomialNB()

In [102]:
nb_model_bow.fit(X_train_bow,y_train)

MultinomialNB()

In [105]:
pred_bow = nb_model_bow.predict(X_test_bow)

In [106]:
print(accuracy_score(y_test, pred_bow))

0.7678125


### **TF - IDF**

In [107]:
tfidf_vectorizer = TfidfVectorizer()

In [110]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test) 

In [109]:
nb2_model = MultinomialNB()

In [111]:
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [112]:
y_pred_tfidf = nb2_model.predict(X_test_tfidf)

In [113]:
print(accuracy_score(y_test, y_pred_tfidf))

0.6609375


## **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

model_logR = LogisticRegression(max_iter=1000)  # more than 100 because we have large sized vectors

In [116]:
model_logR.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [119]:
pred_logR = model_logR.predict(X_test_tfidf)
pred_logR

array([0, 5, 0, ..., 5, 5, 0])

### **Highest Accuracy**

In [118]:
print(accuracy_score(y_test, pred_logR))

0.8615625
